# CURATOR training walkthrough
Use this notebook to prepare a CURATOR training run: compose a Hydra config, inspect/edit key blocks, save the final YAML, then launch training from the CLI. The notebook is lightweight and will not start heavy training jobs.


## Prerequisites
- CURATOR installed (`pip install curator_torch` or `pip install -e .` from the repo) and PyTorch available.
- A dataset in ASE-readable format (e.g., `.traj`). Example uses `curator/example/LiFePO4.traj`.
- GPU recommended (`device=cuda`), CPU works (`device=cpu`). Adjust `trainer.devices` for the number of GPUs/CPUs.
- A writable `run_path` for logs, checkpoints, and compiled models.


## Imports (safe to run)


In [1]:
from curator.utils import read_user_config
from omegaconf import OmegaConf


/home/yangxin/miniconda3/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/yangxin/curator/curator/layer/_cuequivariance_wrapper.py:19: RuntimeWarning: cuequivariance could not be loaded (ModuleNotFoundError("No module named 'cuequivariance.group_theory'")); cueq acceleration disabled.
  warnings.warn(
/home/yangxin/curator/curator/layer/_cuequivariance_wrapper.py:61: UserWarning: cuequivariance is not available. Cuequivariance acceleration will be disabled.
  warnings.warn("cuequivariance is not available. Cuequivariance acceleration will be disabled.")


## 0) Quick CLI smoke test
Fastest way to see CURATOR run: call `curator-train` with just your dataset path and an output folder. We add `fast_dev_run=true` so it finishes in seconds (just a few steps).


In [ ]:
# Minimal on-CPU fast_dev_run smoke test (does only a few steps)
!curator-train \
  data.datapath=../example/LiFePO4.traj \
  run_path=./runs/demo_cli \
  device=cpu trainer.accelerator=cpu \
  trainer.fast_dev_run=true trainer.max_epochs=1

## 0.1) Config file + inline overrides
Write a small user config, then launch with `curator-train cfg=...` while overriding a few parameters from the CLI (still using `fast_dev_run` to keep it quick).


In [ ]:
%%bash
cat > train_quick.yaml <<'YAML'
defaults:
  - trainer: default_trainer
  - task: default_task
  - model: nnp
  - data: custom
  - optional model/repr_params: ${model/representation}_params
  - optional task/output_params: ${task/outputs}_params
  - _self_
data:
  datapath: ../example/LiFePO4.traj
run_path: ./runs/demo_cfg
device: cpu
YAML

# Run with the config and override a few options inline
curator-train cfg=train_quick.yaml

## 0.2) More CLI patterns to try
Pick one of these common patterns. Commands keep `fast_dev_run=true` where possible so you can test safely.

1) Config file + CLI override (fast_dev_run)
```
curator-train   cfg=train_quick.yaml   trainer.fast_dev_run=true   run_path=./runs/demo_cfg_fast
```

2) Multi-GPU example (adjust device count to your hardware)
```
curator-train   cfg=train_quick.yaml   device=cuda trainer.accelerator=cuda trainer.devices=2   run_path=./runs/demo_multi_gpu   trainer.max_epochs=2
```

3) Switch presets via defaults (pick another preset)
```
curator-train   cfg=train_quick.yaml   defaults=[trainer=default_trainer,model=nnp,data=custom]   run_path=./runs/demo_defaults
```


## 1) Load and compose the config
Set `cfg_path` to your user YAML. `read_user_config` merges it with defaults from `curator/curator/configs/train.yaml`, resolves interpolations, and returns a single `DictConfig` you can inspect.


In [2]:
cfg_path = '../example/train/config.yaml'  # change to your own user config
cfg = read_user_config(cfg_path)

In [16]:
model = instantiate(cfg.model)
datamodule = instantiate(cfg.data)

## 2) Hydra-style config basics (what happens under the hood)
Hydra composes configs in layers: it starts from defaults, merges user files, then applies CLI overrides. Below are the core ideas with short examples.

- **Defaults tree**: `curator/curator/configs/train.yaml` pulls in trainer/task/model/data presets so you get sensible starting values. A user file (e.g., `train_quick.yaml`) can override any key. Example: `data.datapath: ../example/LiFePO4.traj` inside your YAML replaces the default dataset path.

- **Interpolation**: You can reference other fields to avoid duplication. `${run_path}` or `${task.energy_weight}` will be substituted after composition. Example: callbacks save to `${run_path}/model_path` so changing `run_path` updates all downstream paths automatically.

- **CLI overrides (highest priority)**: `key=value` pairs on the command line apply after all files. Example: `curator-train cfg=train_quick.yaml task.optimizer.lr=3e-4 trainer.devices=2` bumps learning rate and GPU count without editing the YAML. Quote values with spaces/special chars.

- **Lists/dicts creation**: If a node does not exist, `++path.to.field=value` creates it. Example: `++trainer.callbacks.early_stopping.patience=50` adds/overwrites that field even if it was absent (rare here because defaults are populated).

- **Disabling entries**: Prefix with `~` to drop a default. Example: `~trainer.callbacks.early_stopping` removes that callback when you want uninterrupted training.

- **Switching presets via defaults**: You can swap whole presets without touching files by passing a new defaults list. Example: `defaults=[trainer=default_trainer,model=nnp,data=custom]` keeps the standard trainer/model/data set; you could swap `model=mace` if you have that preset.

- **Resolution order**: base defaults → user config (`cfg=...`) → CLI overrides. Later layers win. If two places set `trainer.devices`, the CLI value is used.

- **Debug tip**: Before running, inspect the composed config to ensure overrides did what you expect:

```python
from curator.utils import read_user_config
from omegaconf import OmegaConf
cfg = read_user_config('train_quick.yaml')
print(OmegaConf.to_yaml(cfg, resolve=False))
```
This prints the final structure (with interpolations unresolved) so you can verify paths, hyperparameters, and callbacks.


## 3) Where to look afterward (and wandb logging)
- `run_path/logs/`: CSV logs from the Lightning trainer.
- `run_path/model_path/`: checkpoints; best model named by validation loss.
- `run_path/compiled_model.pt`: produced when `deploy_model` is true.
- `config.yaml`: the exact config used for the run.
- **wandb runs**: enable `WandbLogger` (e.g., `trainer.logger._target_=pytorch_lightning.loggers.WandbLogger trainer.logger.project=curator-demo trainer.logger.name=run1`). Local wandb files live under `trainer.logger.save_dir`. Install/login first: `pip install wandb && wandb login`.
- **wandb offline mode**: if you lack network access, set env `WANDB_MODE=offline` (or `WANDB_DISABLED=true`) before running. Example: `WANDB_MODE=offline curator-train cfg=config.yaml ...`. After training, sync logs with `wandb sync /path/to/run_path/wandb/latest-run` (or the specific offline run directory) once you regain connectivity.


## 4) How to train a qeq model

In [ ]:
!curator-train model=qeq.yaml data.datapath=../example/LiFePO4.traj trainer.max_epochs=1 run_path=./runs/demo_qeq

## 5) Quick troubleshooting
- **CUDA OOM**: lower batch size, try `precision: 16` if supported, or use CPU for debugging.
- **Data loader errors**: confirm `data.datapath` (or `train_path`/`val_path`) exists; set `num_workers=0` if multiprocessing is an issue.
- **Hydra override errors**: ensure `key=value` formatting; quote strings with special characters.
- **Resume behavior**: `task.load_weights_only=true` loads weights; `task.load_entire_model=true` resumes optimizer state; otherwise set `model_path` as `ckpt_path` via Hydra.
